In [ ]:
import fitz  # PyMuPDF
import os

pdf_path = ['dataset/'+o for o in os.listdir("dataset") if ".pdf" in o]

for pdf in pdf_path:
    txt_path = os.path.join(pdf.replace(".pdf", ".txt"))
    with fitz.open(pdf) as doc, open(txt_path, "w", encoding="utf-8") as out_file:
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text("text")
            out_file.write(f"\n--- Page {page_num} ---\n")
            out_file.write(text.strip())
            out_file.write("\n")

print(f"Text extracted and saved to {txt_path}")

In [ ]:
with open('156_11-Nisan_E.txt', 'r', encoding='utf-8') as file:
    lines = file.read()
    chunks = lines.split('.')

print(len(chunks))

In [ ]:
import base64

# Replace this with your base64 string (omit data URL prefix if present)
with open('audio_base64.txt', 'r') as fl:
    base64_audio = fl.read()   # truncated for example

# Optional: remove base64 prefix if it exists (e.g., "data:audio/wav;base64,")
if base64_audio.startswith("data:"):
    base64_audio = base64_audio.split(",")[1]

# Decode and write to file
audio_bytes = base64.b64decode(base64_audio)

# Save to file (change extension depending on format: .wav, .mp3, etc.)
output_path = "output_audio.wav"
with open(output_path, "wb") as f:
    f.write(audio_bytes)

print(f"Audio saved to {output_path}")


In [ ]:
from datetime import datetime

def convert_srt_time_to_seconds(timestamp):
    """Converts a timestamp like '00:01:23.456' to seconds as a float."""
    dt = datetime.strptime(timestamp.strip(), "%H:%M:%S.%f")
    return dt.minute * 60 + dt.second + dt.microsecond / 1_000_000

def srt_to_audacity_labels(srt_path, output_path):
    with open(srt_path, 'r', encoding='utf-8') as srt_file, \
         open(output_path, 'w', encoding='utf-8') as txt_file:
        
        lines = [line.strip() for line in srt_file]
        i = 0
        while i < len(lines):
            if not lines[i]:  # Skip empty lines
                i += 1
                continue
            
            # Extract start and end times
            if i+1 < len(lines) and '-->' in lines[i+1]:
                start_end = lines[i+1].replace(',', '.').split(' --> ')
                start = convert_srt_time_to_seconds(start_end[0].strip())
                end = convert_srt_time_to_seconds(start_end[1].strip())
                i += 2
            else:
                i += 1
                continue
            
            # Extract multi-line text
            text_lines = []
            while i < len(lines) and lines[i]:
                text_lines.append(lines[i])
                i += 1
            
            # Clean and join label text
            text = ' '.join(text_lines).replace('\t', ' ')
            
            # Write to output in Audacity label format
            txt_file.write(f"{start:.3f}\t{end:.3f}\t{text}\n")
            i += 1

# Example usage:
srt_to_audacity_labels("[sub] Stereo Mix (4).wav.srt", "audacity_labels_4.txt")

In [ ]:
import base64

def save_base64_to_wav(base64_string, output_file_path):
    # Remove any header (e.g., "data:audio/wav;base64,") if present
    if ',' in base64_string:
        base64_string = base64_string.split(',')[1]
    
    audio_data = base64.b64decode(base64_string)

    with open(output_file_path, 'wb') as f:
        f.write(audio_data)
%cd ..
# Example usage
with open('audio_base64.txt', 'r', encoding='utf-8') as fl:
    base64_audio = fl.read() # your base64 string
    save_base64_to_wav(base64_audio, "output.wav")


In [ ]:
import glob
import re
import os

def merge_audacity_labels(input_pattern="audacity_labels_{}.txt",
                          num_files=4,
                          output_path="merged_labels.txt",
                          sep="\t"):
    """
    Merges multiple Audacity label files into one, adjusting timestamps so
    they reflect concatenated audio.

    Parameters
    ----------
    input_pattern : str
        A Python-format pattern for your files, with one '{}' for the index.
    num_files : int
        How many files (1..num_files) to read.
    output_path : str
        Path to write the merged label file.
    sep : str
        Field separator in the label files (usually '\t').
    """
    merged_entries = []
    cumulative_offset = 0.0

    for i in range(1, num_files + 1):
        path = input_pattern.format(i)
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Expected label file not found: {path}")

        # Read & parse
        entries = []
        max_end = 0.0
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(sep)
                if len(parts) < 2:
                    continue  # skip malformed
                start, end = map(float, parts[:2])
                label = parts[2] if len(parts) >= 3 else ""
                # track longest timestamp in this file
                if end > max_end:
                    max_end = end
                entries.append((start, end, label))

        # Offset and append
        for (start, end, label) in entries:
            new_start = start + cumulative_offset
            new_end   = end   + cumulative_offset
            merged_entries.append((new_start, new_end, label))

        # bump offset by this file’s length
        cumulative_offset += max_end

    # Write out sorted by start time (just in case)
    merged_entries.sort(key=lambda x: x[0])

    with open(output_path, 'w', encoding='utf-8') as out:
        for start, end, label in merged_entries:
            out.write(f"{start:.6f}{sep}{end:.6f}{sep}{label}\n")

    print(f"Merged {num_files} files into '{output_path}' "
          f"(total duration ≃ {cumulative_offset:.2f} sec).")


if __name__ == "__main__":
    merge_audacity_labels()


In [ ]:
import os
import re

def rename_wav_files_to_5_digits(directory):
    pattern = re.compile(r'^wav-(\d+)\.wav$')

    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            num = int(match.group(1))
            new_filename = f"wav-{num:05d}.wav"
            old_path = os.path.join(directory, filename)
            new_path = os.path.join(directory, new_filename)

            if old_path != new_path:
                os.rename(old_path, new_path)
                print(f"Renamed: {filename} -> {new_filename}")
            else:
                print(f"Already correct: {filename}")

# Example usage
rename_wav_files_to_5_digits("wavs")


In [ ]:
!ls